<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# TMA load + mbarrier

The **Tensor Memory Accelerator (TMA)** is a hardware copy engine that moves an entire tile from
global memory into shared memory with one instruction. No per-thread address math, no element loop:
one elected thread fires the copy, and the engine streams the bytes in the background while the rest
of the kernel runs.

This is the smallest complete TMA program. The host builds a *descriptor*, one thread fires one
async bulk copy, every thread waits on an **mbarrier**, and we copy the loaded tile back to global
memory to check it against PyTorch. That handshake — descriptor, copy, mbarrier wait — is the
building block under every high-performance GEMM and attention kernel later in this course.

**You'll learn:** how to build a TMA descriptor (`TensorMap`) on the host and pass it as a
`cutlass.GridConstant`; the full mbarrier handshake for one async copy
(`init → fence → barrier → arrive_expect_tx → cp_async → try_wait_parity`); why an mbarrier counts
*bytes*, not threads; and the `elect_sync` single-lane pattern for issuing the copy.

**Runs on:** Hopper (sm_90) or newer — TMA and bulk-tensor copies are sm_90+ only. No GPU? The
kernel still compiles under `CUTE_DSL_DRYRUN=1`; the numeric check needs real hardware.
**Prereq:** the `01_array_concepts` and `02_vector_concepts` notebooks.

## 1. Why TMA, and what is a descriptor?

In the `01_array_concepts` notebook every thread computed its own global address and loaded one
element into shared memory. TMA collapses that into **one instruction**: you hand the engine a
small struct describing the global tensor and the tile to copy, and it does the rest. Two new
objects appear:

- **TMA descriptor (`TensorMap`)** — a host-built struct describing the global tensor (shape,
  strides, the tile `box` size, and the SMEM swizzle). The engine reads it to find the bytes, so
  your kernel never computes an address. It is passed as `cutlass.GridConstant[cuda.TensorMap]`, so
  it lives in constant memory — one copy for the whole grid.
- **mbarrier** — an `Int64` in shared memory that counts *bytes still in flight*; a copy "arrives"
  once its bytes land. It is **not** a `cute.arch.barrier`, which synchronizes *threads*.

The handshake for one async copy reads top to bottom:

```text
  mbarrier_init(mbar, 1)             one elected thread sets the mbarrier up
  fence_mbarrier_init(); barrier     make the init visible to all threads
  mbarrier_arrive_expect_tx(bytes)   tell the mbarrier how many bytes are coming
  cp_async_bulk_tensor(...)          fire the TMA copy (global -> SMEM)
  while not try_wait_parity(...)     every thread spins until the bytes have landed
```

First the imports. We build the descriptor with `cutlass.experimental.cuda` (imported as `cuda`),
annotate the kernel's descriptor argument with `cutlass.GridConstant`, and pull the mbarrier /
bulk-copy intrinsics from `cutlass.experimental.primitives` (imported as `prims`). The tile
size — a 64x64 fp16 tile is 8 KiB of SMEM — is set later, in the run cell.

In [ ]:
import cutlass
from cutlass.experimental import primitives as prims  # mbarrier + TMA bulk-copy + barrier intrinsics
import cutlass.cute as cute
import cutlass.experimental.cuda as cuda

import torch
import math

## 2. The kernel

The kernel takes the descriptor and a global output tensor. It allocates the SMEM tile and a
one-entry mbarrier, runs the handshake, fires one TMA copy of the tile at global coordinate
`(0, 0)`, waits for it to land, and copies the loaded tile back to global memory.

Three details to note:

- **`prims.elect_sync()` picks exactly one lane.** A single TMA copy needs only one issuing thread;
  the engine moves the whole tile regardless.
- **`arrive_expect_tx` declares the exact byte count** (`box elements × 2 bytes` for fp16). Without
  it the wait never reaches the right count and the kernel hangs.
- **The two synchronizations do different jobs, and you need both.** `cute.arch.barrier()` waits on
  *threads* — it makes the mbarrier init visible to every thread before anyone uses it. The
  `try_wait_parity` spin waits on the *mbarrier* — for the copy's bytes to fully land.

The final write-back is a plain cooperative copy — 32 threads each own two columns of the 64-wide
tile across all 64 rows — and exists only so the host can verify the load.

In [ ]:
@cute.kernel
def tma_load_kernel(
    tma_desc: cutlass.GridConstant[cuda.TensorMap],
    out_tensor: cutlass.Array,
    box_dim: cutlass.Constexpr,
    TILE: cutlass.Constexpr = 64,
):
    lane, _, _ = cute.arch.thread_idx()

    # Step 1. Allocate the SMEM destination tile and a 1-entry mbarrier for the copy.
    smem_tile = cutlass.Array(cutlass.Float16, (TILE, TILE), space=cutlass.AddressSpace.smem)
    mbar = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)

    # Step 2. One lane prefetches the descriptor and arms the mbarrier for a single arrival.
    if prims.elect_sync():
        prims.prefetch_tensormap(tma_desc.get_ptr())
        prims.mbarrier_init(mbar, 1)

    # Step 3. Publish the mbarrier init to every thread before anyone waits on it.
    prims.fence_mbarrier_init()
    cute.arch.barrier()

    # Step 4. One lane declares the incoming byte count, then fires the bulk copy of the
    # tile at global coordinate (0, 0). arrive_expect_tx wants a byte count, so shift
    # bits to bytes with `// 8`.
    if prims.elect_sync():
        tile_bytes = math.prod(box_dim) * cutlass.Float16.width // 8
        prims.mbarrier_arrive_expect_tx(mbar, tile_bytes)
        prims.cp_async_bulk_tensor_shared_cta_global(
            smem_tile, tma_desc.get_ptr(), (0, 0), mbar
        )

    # Step 5. Every thread spins until the tile has fully landed. The timelimit variant
    # retries on a tick-timeout and uses `.acquire.cta` ordering, so the TMA writes are
    # visible to all threads once the wait returns.
    while not prims.mbarrier_try_wait_parity(mbar, 0):
        pass

    # Step 6. Cooperative write-back SMEM -> global: lane owns columns `lane` and `lane + 32`
    # across all 64 rows. Verification only.
    for r in range(TILE):
        out_tensor[r, lane] = smem_tile[r, lane]
        out_tensor[r, lane + 32] = smem_tile[r, lane + 32]

## 3. Host: build the descriptor and launch

`create_tensor_map_tiled_from_view` is duck-typed: it reads the source's
`shape` / `strides` / `dtype` / `data_ptr()` to program the engine, so a flat `cutlass.Array` (what
`from_dlpack` produces) feeds it directly — no separate layout object. Three arguments shape the copy:

| argument | effect |
|---|---|
| `box_dims=box_dim[::-1]` | tile size in TMA (descriptor) order — the *reverse* of logical order |
| `stride_order=(1, 0)` | marks the innermost (contiguous) logical dimension, so kernel coordinates and the descriptor agree on the contiguous axis |
| `swizzle=cuda.TensorMapSwizzle.none` | lays the SMEM tile out exactly like the logical tile, so `smem_tile[r, c] == matrix[r, c]` and the readback compares directly against PyTorch |

(`s128b` swizzle permutes the SMEM layout for bank-conflict-free tensor-core access — an
optimization we defer to the GEMM chapters, where the consumer is swizzle-aware.)

In [ ]:
@cute.jit
def tma_load(matrix: cutlass.Array, out_tensor: cutlass.Array, TILE: cutlass.Constexpr = 64):
    # Build the descriptor straight off `matrix` (shape/strides/dtype/data_ptr).
    box_dim = (TILE, TILE)
    tma_desc = cuda.create_tensor_map_tiled_from_view(
        matrix,
        box_dims=box_dim[::-1],
        stride_order=(1, 0),
        swizzle=cuda.TensorMapSwizzle.none,
    )

    # One CTA of 32 threads is enough: a single lane issues the copy.
    tma_load_kernel(tma_desc, out_tensor, box_dim, TILE).launch(
        grid=(1, 1, 1), block=(32, 1, 1)
    )

## 4. Run it and verify

We build a 128x128 source whose value at `[r, c]` is `r*100 + c`, TMA the top-left 64x64 tile into
SMEM, copy it back to a global output, and compare against the same slice in PyTorch.

With `swizzle=none` and `stride_order=(1, 0)`, the loaded tile is the top-left 64x64 block in
**natural (non-transposed) order**: `got[r, c] == src[r, c]`. Row 0 reads `0, 1, 2, ...` and
column 0 reads `0, 100, 200, ...`.

One caveat: pull the result to the host with `.cpu()` *before* comparing, and compute the
reference host-side so all comparison math runs on CPU tensors. Boolean indexing and
`assert_close` are most predictable on CPU tensors.

In [ ]:
# Source value at [r, c] is r*100 + c, so each row and column is easy to eyeball.
# Keep a CPU copy for the reference and allocate a 64x64 output for the loaded tile.
rows, cols = 128, 128
src_cpu = torch.arange(cols).unsqueeze(0) + (torch.arange(rows) * 100.0).unsqueeze(1)
src_cpu = src_cpu.to(torch.float16)
src = src_cpu.cuda()
TILE = 64
out = torch.zeros(TILE, TILE, dtype=torch.float16, device="cuda")

torch.set_printoptions(precision=1, sci_mode=False, linewidth=120)

# Pass both torch tensors through from_dlpack as cutlass.Array. The descriptor is
# built from the static 128x128 layout, exactly what a fixed-size tutorial wants:
# the compiler bakes the concrete shape and strides into the TMA descriptor.
tma_load(cute.runtime.from_dlpack(src), cute.runtime.from_dlpack(out), TILE)

# Verify on the host: pull to CPU first, then compare CPU tensors (see note above).
# The loaded tile is the top-left 64x64 block of `src`, in natural order.
got = out.cpu()
ref = src_cpu[:TILE, :TILE].contiguous()
print("loaded tile, row 0:", got[0, :8].tolist())
print("loaded tile, col 0:", got[:8, 0].tolist())
assert bool(torch.equal(got, ref)), "TMA-loaded tile does not match the reference"

print("PASS")

# Expected output (value at [r, c] is r*100 + c, loaded in natural order):
# loaded tile, row 0: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
# loaded tile, col 0: [0.0, 100.0, 200.0, 300.0, 400.0, 500.0, 600.0, 700.0]
# PASS

## Try it yourself

1. Change the copy coordinate from `(0, 0)` to `(0, 64)` in the kernel — which 64x64 tile of `src`
   lands in SMEM now, and how should the reference slice change to match?
2. Delete the `mbarrier_arrive_expect_tx` line. The wait can no longer reach the right byte count —
   what happens, and why does every TMA copy pair with an expect-tx?
3. The mbarrier (an `Int64` in SMEM) tracks *bytes in flight*, while `cute.arch.barrier` syncs
   *threads*. Explain in one sentence why this kernel needs both.
4. Switch `swizzle` to `cuda.TensorMapSwizzle.s128b` and rerun. The numeric check fails because the
   SMEM bytes are now permuted — a preview of why GEMM kernels keep a swizzle-aware consumer.

## Peeking under the hood

Set a `CUTE_DSL_*` option before running the launch cell — in a fresh cell run
`import os; os.environ["CUTE_DSL_PRINT_PTX"] = "1"`, then re-execute the run cell — to see the
IR / PTX the DSL generates:

- `CUTE_DSL_DRYRUN=1` — trace and compile only, no GPU
- `CUTE_DSL_PRINT_IR=1` — the generated IR (look for the `cp.async.bulk` op)
- `CUTE_DSL_PRINT_PTX=1` — PTX (`cp.async.bulk.tensor` + mbarrier ops)